# 01 — Cleaning

**Owner:** Shaik Tajuddin · **Phase 1, step 2**

Turns `data/raw/Combined_dataset.csv` into `data/processed/clean.csv`.

Every decision below is recorded **with its reason**, because the viva is individual
and examiners ask *why*, not just *what*. The logic itself lives in `src/clean.py` so
that it is importable and testable; this notebook is the audit trail.

Rules from the Phase 1 work plan:

1. make units consistent
2. handle detection-limit values such as `<0.5` with **one stated rule**
3. flag impossible values instead of silently deleting them
4. remove exact duplicates
5. **no imputation here** — it belongs in `src/preprocess.py`, fitted on train only


In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd

from src import clean as C
from src.config import RAW_CSV, CLEAN_CSV, TARGET_COLUMN, TARGET_KIND

pd.set_option('display.width', 120)
print(f'target: {TARGET_COLUMN}  ({TARGET_KIND})')
print(f'raw:    {RAW_CSV}')


target: CCME_Values  (regression)
raw:    /Users/Taj786/Library/Mobile Documents/com~apple~CloudDocs/projects/water_quality_prediction/water_quality_prediction/data/raw/Combined_dataset.csv


## 1. Load the raw file

Three choices worth defending:

- **`Area` read as a category.** 61,689 unique values over 2.8M rows; as plain objects
  the frame costs roughly 1.5 GB, as categories about 400 MB.
- **`Date` parsed with `errors="raise"`.** All 2,827,977 values match `%d-%m-%Y`.
  Raising rather than coercing means a future data refresh fails loudly instead of
  silently filling the column with `NaT`.
- **Column names kept verbatim.** `src/config.py` binds `TARGET_COLUMN = "CCME_Values"`,
  so renaming would break Parth's interface and every downstream module.


In [2]:
df = C.load_raw()
print(f'{len(df):,} rows x {df.shape[1]} columns')
df.head(3)


2,827,977 rows x 15 columns


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),Nitrogen (mg/l),Nitrate (mg/l),CCME_Values,CCME_WQI,Date_parsed
0,Canada,SE649035-145565,River,12-01-1974,0.059248,1.30,8.15,0.011917,8.07500,9.885,0.343917,11.73155,100.0,Excellent,1974-01-12
1,Canada,SE649035-145565,River,12-01-1975,0.039821,1.38,7.80,0.009417,7.73333,10.150,0.449083,11.82009,100.0,Excellent,1975-01-12
2,Canada,SE649035-145565,River,12-01-1976,0.031341,2.23,7.80,0.011000,7.46667,10.235,0.220750,14.87472,100.0,Excellent,1976-01-12


In [3]:
# Missing values, as delivered

missing = df.isna().sum()
print(missing.to_string())
print(f'\ntotal missing: {missing.sum():,}')


Country                             0
Area                                0
Waterbody Type                      0
Date                                0
Ammonia (mg/l)                      0
Biochemical Oxygen Demand (mg/l)    0
Dissolved Oxygen (mg/l)             0
Orthophosphate (mg/l)               0
pH (ph units)                       0
Temperature (cel)                   0
Nitrogen (mg/l)                     0
Nitrate (mg/l)                      0
CCME_Values                         0
CCME_WQI                            0
Date_parsed                         0

total missing: 0


**Zero missing values — do not celebrate.** Section 4 shows why: the gaps were filled
before the dataset reached us.


## 2. Units

**Decision: no conversion applied.**

**Reason:** each measurement column already carries one unit, declared in its own name
(`mg/l`, `ph units`, `cel`). The ranges below are consistent with those units — no
column shows the bimodal spread you would see if, say, mg/l and µg/l rows were mixed.
Converting anything here would be inventing a problem.


In [4]:
df[C.MEASUREMENTS].describe().T[['min','25%','50%','75%','max']]


,min,25%,50%,75%,max
Ammonia (mg/l),-0.005,0.030,0.055,0.317,200.0
Biochemical Oxygen Demand (mg/l),-2.000,1.600,2.700,2.830,255.0
Dissolved Oxygen (mg/l),0.000,9.860,10.200,11.000,20.0
Orthophosphate (mg/l),-0.004,0.040,0.107,0.227,100.0
pH (ph units),-1.000,7.550,7.780,8.000,30.0
Temperature (cel),-5.260,8.990,11.460,14.200,98.0
Nitrogen (mg/l),0.000,0.780,4.000,6.320,46.0
Nitrate (mg/l),0.000,1.173,4.500,4.940,155.0


## 3. Detection limits

**Decision: no rule applied — none exist.**

**Reason:** 5,508 rows in the file contain a `<` character, and every one of them is
inside an `Area` site name, for example `BURNHAM MARKET STW F/E <=2028M3/D`. All eight
measurement columns parse as pure floats. The upstream harmonization already resolved
censored values, by means we cannot inspect.

The check below stays in the code rather than becoming a comment: if a data refresh
introduces detection limits, it reports them instead of coercing them to `NaN` unseen.


In [5]:
print('censored values in measurement columns:', C.check_detection_limits(df) or 0)

# Where the '<' characters actually are
area_hits = df['Area'].astype(str).str.contains('<')
print(f"'<' inside Area names: {area_hits.sum():,} rows")
df.loc[area_hits, 'Area'].drop_duplicates().head(5).tolist()


censored values in measurement columns: 0
'<' inside Area names: 5,508 rows


['BURNHAM MARKET STW F/E  <=2028M3/D',
 'FOXTON CAMBS STW F/E <=3159M3/D',
 'DORRINGTON STW F/E FLOW <2.05 L/S',
 'BRAUNSTON STW F/E <=8.11L/S',
 'ALCONBURY STW F/E FLOW <15.7 L/S']

## 4. Upstream imputation — the main finding

**Decision: flag, do not repair.**

**Reason:** the file reports no missing values, but five of the eight measurement
columns carry their own **exact median** as the single most frequent value, at 11–32 %
of rows. No natural measurement distribution does that. It is the signature of a median
fill applied upstream.

We cannot undo it — the original gaps are unrecoverable — so we mark it and let Parth's
pipeline and Jivit's ablation measure the cost, instead of inheriting it silently.

This also means the project rule *"imputation is fitted on the training split only"* is
**already broken by the data as delivered**: the upstream fill used medians computed
over every row, including rows that will become our test set.


In [6]:
for col in C.MEASUREMENTS:
    counts = df[col].value_counts()
    mode, n = counts.index[0], counts.iloc[0]
    med = df[col].median()
    sig = ' <-- equals median' if np.isclose(mode, med) else ''
    print(f'{col:36s} mode={mode:<8g} {n:>9,} ({100*n/len(df):5.2f}%)  median={med:<8g}{sig}')


Ammonia (mg/l)                       mode=0.03       526,716 (18.63%)  median=0.055   
Biochemical Oxygen Demand (mg/l)     mode=2.7        686,753 (24.28%)  median=2.7      <-- equals median
Dissolved Oxygen (mg/l)              mode=10.2       893,535 (31.60%)  median=10.2     <-- equals median
Orthophosphate (mg/l)                mode=0.144      444,309 (15.71%)  median=0.107   
pH (ph units)                        mode=7.78       367,499 (13.00%)  median=7.78     <-- equals median
Temperature (cel)                    mode=11.46      311,639 (11.02%)  median=11.46    <-- equals median
Nitrogen (mg/l)                      mode=5          409,140 (14.47%)  median=4       
Nitrate (mg/l)                       mode=4.5        716,630 (25.34%)  median=4.5      <-- equals median


In [7]:
df, imputation = C.flag_upstream_imputation(df)
print(f"columns flagged: {len(imputation)} of {len(C.MEASUREMENTS)}")
print(f"median signature: {sum(i['equals_median'] for i in imputation.values())}")

dist = df['n_imputed_upstream'].value_counts().sort_index()
pd.DataFrame({'rows': dist, 'share %': (100*dist/len(df)).round(2)})


columns flagged: 8 of 8
median signature: 5


,rows,share %
n_imputed_upstream,,
0,1110647,39.27
1,602535,21.31
2,408207,14.43
3,190571,6.74
4,248878,8.80
5,233481,8.26
6,32972,1.17
7,678,0.02
8,8,0.00


### Why this is a modelling problem, not a cleaning problem

The fill is not spread evenly across the dataset — it is almost entirely England, which
is 75.3 % of the rows. Two consequences:

1. `n_imputed_upstream` is a near-proxy for `Country == England`, so any model given
   these flags can recover country membership from them.
2. The target moves with the flag. Fill status and water quality are **confounded**, so
   a model can score well by detecting which country filled its gaps rather than by
   learning chemistry. A site-separated split will not fix this, because the confound
   follows the sites.

Carried into `02_eda.ipynb` and the report's Limitations section.


In [8]:
by_country = df.groupby('Country', observed=True)['n_imputed_upstream'].agg(['mean','count'])
display(by_country.sort_values('mean', ascending=False).round(3))

by_flag = df.groupby('n_imputed_upstream', observed=True)[TARGET_COLUMN].agg(['mean','count'])
display(by_flag.round(2))


,mean,count
Country,,
England,2.038,2129198
Ireland,0.040,235019
China,0.016,45997
USA,0.014,413814
Canada,0.008,3949


,mean,count
n_imputed_upstream,,
0,91.37,1110647
1,83.36,602535
2,78.49,408207
3,75.89,190571
4,81.81,248878
5,81.72,233481
6,84.63,32972
7,89.15,678
8,89.63,8


## 5. Impossible values

**Decision: flag every impossible reading, delete no rows.**

**Reason:** the work plan is explicit — deleting them would hide a data-quality problem
the report has to discuss. Flags let Parth drop or keep them at pipeline time, which is
a modelling decision rather than a cleaning one.

| Flag | Rule | Why that threshold |
| :--- | :--- | :--- |
| `flag_ph_out_of_range` | pH outside 0–14 | the pH scale has no values beyond it; observed −1.0 to 30.0 |
| `flag_negative_measurement` | any negative non-temperature measurement | a concentration cannot be below zero |
| `flag_temp_implausible` | temperature above 40 °C | observed max 98 °C; surface water does not reach that |
| `flag_temp_below_zero` | temperature below 0 °C | **flagged for review, not called impossible** — saline water can be sub-zero |


In [9]:
df, counts = C.flag_impossible_values(df)
for label, n in counts.items():
    print(f'{label:42s} {n:>8,}')


pH outside 0-14                                 505
negative concentration                           15
temperature above 40 C                        7,284
temperature below 0 C (review only)              32
rows with any impossible value                7,676


In [10]:
# What the flagged rows actually look like

cols = ['Country','Waterbody Type','pH (ph units)','Temperature (cel)', TARGET_COLUMN]
df.loc[df['flag_any_impossible'], cols].head(8)


,Country,Waterbody Type,pH (ph units),Temperature (cel),CCME_Values
221,Canada,Lake,19.44500,14.266700,91.190735
1108,Canada,River,7.80833,41.666667,85.823351
1307,Canada,River,7.79000,89.000000,77.209766
3802,Canada,Lake,19.44500,14.266700,91.190735
36343,England,Effluent,11.36000,41.900000,58.416615
45289,England,River,8.45000,92.500000,87.400049
45771,England,River,7.92000,94.000000,87.238376
48665,England,River,7.81000,96.400000,78.736355


## 6. Exact duplicates

**Decision: remove, keeping the first occurrence.**

**Reason:** these rows are identical across all 14 original columns, so they carry no
information. Left in, they would also corrupt any split — the same observation could
land in both train and test, which is leakage by another name.

Compared on the original columns only, so that "duplicate" stays defined by the source
data rather than by the flags we just derived.


In [11]:
df, removed = C.drop_exact_duplicates(df)
print(f'removed {removed:,} exact duplicate rows')
print(f'remaining: {len(df):,} rows x {df.shape[1]} columns')


removed 9,971 exact duplicate rows
remaining: 2,818,006 rows x 30 columns


## 7. No imputation here

**Decision: none, by design.**

**Reason:** imputation must be fitted on the training split only, and the split does not
exist yet — it is Parth's `src/split.py`. Imputing now would fit statistics across rows
destined for the test set, which is exactly the leakage the upstream pipeline already
committed on our behalf.


In [12]:
assert df.isna().sum().sum() == 0, 'unexpected missing values after cleaning'
print('no missing values introduced by cleaning')


no missing values introduced by cleaning


## 8. Write the output

`data/processed/clean.csv` is **git-ignored**. Teammates regenerate it with
`python -m src.clean` — the pipeline is deterministic, so everyone gets the same file.


In [13]:
from src.config import DATA_PROCESSED
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
df.to_csv(CLEAN_CSV, index=False)
print(f'wrote {CLEAN_CSV.name}  ({CLEAN_CSV.stat().st_size/1e6:.0f} MB)')
print(f'{len(df):,} rows x {df.shape[1]} columns')


wrote clean.csv  (586 MB)
2,818,006 rows x 30 columns


## Summary of decisions

| # | Decision | Reason |
| :-- | :--- | :--- |
| 1 | Column names kept verbatim | `src/config.py` binds `CCME_Values`; renaming breaks the shared interface |
| 2 | `Date` parsed once, original string kept | Parth's split may key on time; one parse avoids five inconsistent ones |
| 3 | No unit conversion | every column already carries a single declared unit |
| 4 | No detection-limit rule | none exist — the `<` characters are all in `Area` names |
| 5 | Upstream imputation flagged, not repaired | the original gaps are unrecoverable; flagging lets the team measure the cost |
| 6 | Impossible values flagged, rows kept | deleting them would hide a problem the report must discuss |
| 7 | `flag_temp_below_zero` separated from impossible | sub-zero saline water is physically real |
| 8 | Exact duplicates removed | no information, and they would leak across any split |
| 9 | No imputation | belongs in `preprocess.py`, fitted on train only |

**Handover to Parth:** `data/processed/clean.csv`, 2,818,006 rows × 30 columns, no
missing values. `src/evaluate_api.load_data()` reads it unchanged.

**Open item:** dataset source URL and license are still missing from
`docs/data_dictionary.md` — needed from Parth for the report citation.
